In [ ]:
!pip install pytrends
!pip install --upgrade pytrends

In [ ]:
import time
import random
import pandas as pd
from pytrends.request import TrendReq

# ---------------------------------------------------------------------------
# 1. Αρχικοποίηση
#    tz σε λεπτά από UTC. Η Ελλάδα είναι UTC+2 το χειμώνα (120) και UTC+3 το
#    καλοκαίρι (180). Για εβδομαδιαία δεδομένα 12 μηνών δεν επηρεάζει το σχήμα,
#    οπότε το 120 είναι μια χαρα ως σταθερά.
# ---------------------------------------------------------------------------
pytrends = TrendReq(hl='el-GR', tz=120)

# 2. Σταθερός Anchor (μεσαίος όγκος, σπάνια πέφτει στο 0 σε εβδομαδιαία βάση)
anchor = "Adidas Samba"

# 3. Λίστα παπουτσιών
all_shoes = [
    "Nike Air Force", "Nike Dunk", "Air Jordan", "Jordan 4", "Jordan 1", "Jordan 11",
    "Nike TN", "Nike Vomero", "Nike Pegasus", "Nike Cortez", "Nike Blazer",
    "Nike V2K Run", "Nike P-6000", "Nike Air Max", "Nike Air Max 95", "Nike Air Max 97",
    "Nike Air Max 270", "Nike Invincible", "Nike M2K Tekno", "Nike Shox", "Nike Zoom Fly",
    "Adidas Gazelle", "Adidas Campus", "Campus 00s", "Adidas Stan Smith",
    "Adidas Superstar", "Handball Spezial", "Adidas SL 72", "Adidas Ultraboost",
    "Adidas NMD", "Adidas Forum", "Adidas Ozelia", "Adidas Ozweego", "Adidas Astir",
    "Adidas Response CL", "Adidas Continental 80", "Adidas ZX", "Adidas Nizza",
    "New Balance 550", "New Balance 530", "New Balance 2002R", "New Balance 9060",
    "New Balance 327", "New Balance 574", "New Balance 1906R", "New Balance 990",
    "New Balance 1080", "New Balance 610", "New Balance 860", "New Balance XC72",
    "Asics Kayano", "Asics Nimbus", "Asics Cumulus", "Asics GT 2160",
    "Asics Gel 1130", "Asics Gel Lyte", "Asics Novablast", "Asics Quantum", "Asics Trabuco",
    "Puma Palermo", "Puma Mayze", "Puma Speedcat", "Puma RS-X",
    "Puma Suede", "Puma Cali", "Puma Smash", "Puma Carina",
    "Vans Old Skool", "Vans Slip On", "Vans Sk8 Hi", "Vans Knu Skool",
    "Converse All Star", "Converse Run Star Hike", "Converse Chuck 70", "Converse Platform",
    "Hoka Clifton", "Hoka Bondi", "On Cloud", "On Cloudmonster",
    "Salomon XT-6", "Salomon Speedcross", "Mizuno Wave Rider",
    "Crocs", "Ugg Ultra Mini", "Ugg Tasman", "Birkenstock Boston", "Birkenstock Arizona",
    "Dr Martens 1460", "Dr Martens Jadon", "Timberland 6 Inch",
    "Reebok Club C", "Reebok Classic", "Fila Disruptor", "Havaianas", "ugg παντοφλες"
]

# Αφαίρεση του anchor από τη λίστα + αλφαβητική ταξινόμηση
all_shoes = sorted(set(s for s in all_shoes if s != anchor))

_end = date.today()
_start2 = _end - relativedelta(years=2)

geo_target = 'GR'
timeframe_target = f"{_start2} {_end}"

print(f"🚀 Συλλογή για {len(all_shoes) + 1} παπούτσια (Anchor: '{anchor}')...")
print("-" * 50)

# ---------------------------------------------------------------------------
# 4. Master DataFrame: ο anchor μόνος του ορίζει τη βάση της κλίμακας.
# ---------------------------------------------------------------------------
pytrends.build_payload([anchor], timeframe=timeframe_target, geo=geo_target)
master_df = pytrends.interest_over_time()
if master_df.empty:
    raise RuntimeError(f"Ο anchor '{anchor}' επέστρεψε άδειο αποτέλεσμα. Διάλεξε άλλον anchor.")
master_df = master_df.drop(columns=['isPartial'], errors='ignore')

anchor_master_mean = master_df[anchor].mean()
if anchor_master_mean <= 0:
    raise RuntimeError(f"Ο anchor '{anchor}' έχει μηδενικό μέσο όρο — ακατάλληλος για κλιμάκωση.")
print(f"ℹ️ Anchor '{anchor}' μέσος όρος στο master: {anchor_master_mean:.2f}\n")

# 5. Groups των 3 (+ anchor = 4, εντός του ορίου των 5 keywords)
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

shoe_groups = list(chunks(all_shoes, 3))
missing = []

# ---------------------------------------------------------------------------
# 6. Loop με exponential backoff retry
# ---------------------------------------------------------------------------
MAX_RETRIES = 10

for idx, group in enumerate(shoe_groups):
    current_keywords = [anchor] + group
    print(f"📊 Group {idx + 1}/{len(shoe_groups)}: {group}")

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            pytrends.build_payload(current_keywords, timeframe=timeframe_target, geo=geo_target)
            group_df = pytrends.interest_over_time()

            if group_df.empty or anchor not in group_df.columns:
                raise RuntimeError("Άδειο group ή λείπει ο anchor")

            group_df = group_df.drop(columns=['isPartial'], errors='ignore')

            # --- ANCHOR SCALING ---
            # Ο anchor κανονικοποιείται διαφορετικά σε κάθε payload (το max
            # εξαρτάται από το πιο δημοφιλές keyword της ομάδας). Φέρνουμε τον
            # μέσο όρο του anchor αυτής της ομάδας πίσω στον master μέσο όρο
            # και εφαρμόζουμε τον ΙΔΙΟ συντελεστή σε όλα τα παπούτσια της ομάδας.
            anchor_group_mean = group_df[anchor].mean()

            # Όριο >= 1 (όχι απλώς > 0): αν ο anchor είναι σχεδόν μηδενικός εδώ,
            # ο συντελεστής εκρήγνυται και η ομάδα γίνεται αναξιόπιστη.
            if anchor_group_mean < 1:
                raise RuntimeError(
                    f"Anchor σχεδόν μηδενικός σε αυτό το group (mean={anchor_group_mean:.3f}) — αναξιόπιστο"
                )

            scale_factor = anchor_master_mean / anchor_group_mean

            for shoe in group:
                if shoe in group_df.columns:
                    scaled = (group_df[shoe] * scale_factor).round(1)
                    # Ευθυγράμμιση στον index του master ώστε να μην μπουν NaN
                    # σε περίπτωση που το group γύρισε ελαφρώς διαφορετικές εβδομάδες.
                    master_df[shoe] = scaled.reindex(master_df.index)
                else:
                    missing.append(shoe)

            time.sleep(random.uniform(6, 10))
            break  # επιτυχία -> έξοδος από το retry loop

        except Exception as e:
            is_rate = "429" in str(e) or "rate" in str(e).lower()
            base = 20 if is_rate else 10  # 429 -> μεγαλύτερο cool-down
            wait = min(120, base * (2 ** (attempt - 1))) + random.uniform(0, 5)
            print(f"    ⚠️ Προσπάθεια {attempt}/{MAX_RETRIES} απέτυχε: {e}")
            if attempt < MAX_RETRIES:
                print(f"    ⏳ Cool-down {wait:.0f}s και ξαναπροσπάθεια...")
                time.sleep(wait)
            else:
                print(f"    ❌ Group απέτυχε μετά από {MAX_RETRIES} προσπάθειες — μπαίνει στα missing.")
                missing.extend(group)

# ---------------------------------------------------------------------------
# 7. Τακτοποίηση στηλών: Anchor πρώτος
# ---------------------------------------------------------------------------
ordered = [anchor] + [c for c in master_df.columns if c != anchor]
master_df = master_df[ordered]

# 8. Αναφορά για όρους που δεν συλλέχθηκαν (χωρίς διπλότυπα, αλφαβητικά)
missing = sorted(set(m for m in missing if m not in master_df.columns))
if missing:
    print("\n⚠️ Όροι που ΔΕΝ συλλέχθηκαν:")
    for m in missing:
        print("   -", m)

# 9. Αποθήκευση σε CSV
csv_filename = "Ultimate_100_Shoes_Weekly_Greece.csv"
master_df.to_csv(csv_filename, encoding='utf-8-sig')
print("\n" + "=" * 50)
print("🎉 ΟΛΟΚΛΗΡΩΘΗΚΕ!")
print(f"📂 Το αρχείο '{csv_filename}' είναι έτοιμο.")
print(f"   Στήλες: {len(master_df.columns)} | Γραμμές (εβδομάδες): {len(master_df)}")
print("=" * 50)

🚀 Συλλογή για 96 παπούτσια (Anchor: 'Adidas Samba')...
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


ℹ️ Anchor 'Adidas Samba' μέσος όρος στο master: 53.05

📊 Group 1/32: ['Adidas Astir', 'Adidas Campus', 'Adidas Continental 80']
    ⚠️ Προσπάθεια 1/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 24s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 2/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 45s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 3/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 83s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 4/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 125s και ξαναπροσπάθεια...


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


📊 Group 2/32: ['Adidas Forum', 'Adidas Gazelle', 'Adidas NMD']


...

📊 Group 17/32: ['New Balance 530', 'New Balance 550', 'New Balance 574']


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


📊 Group 18/32: ['New Balance 610', 'New Balance 860', 'New Balance 9060']


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


📊 Group 19/32: ['New Balance 990', 'New Balance XC72', 'Nike Air Force']
    ⚠️ Προσπάθεια 1/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 20s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 2/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 43s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 3/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 83s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 4/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 125s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 5/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 124s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 6/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 121s και ξαναπροσπάθεια...
    ⚠️ Προσπάθεια 7/10 απέτυχε: The request failed: Google returned a response with code 429
    ⏳ Cool-down 124s 

In [ ]:
import os, time, random
import pandas as pd
from pytrends.request import TrendReq

# ============================================================
# ΤΕΛΙΚΟΣ ΚΩΔΙΚΑΣ — Brand collection (Google Trends, GR)  [RESUMABLE]
# Μέθοδος: STAR ANCHORING (Nike + Adidas anchors σε ΚΑΘΕ batch).
# Ανθεκτικό στα 429: checkpoint μετά από ΚΑΘΕ batch -> αν κολλήσει,
# απλά ΞΑΝΑΤΡΕΞΕ το cell και συνεχίζει από εκεί που έμεινε.
# ============================================================

# >>> Διάλεξε τι θα τρέξεις. Καλύτερα ΕΝΑ timeframe τη φορά. <<<
RUN = ["1y"]          # π.χ. ["5y"]  ή  ["1y","5y"]
TIMEFRAMES = {"1y": "today 2-my", "5y": "today 5-y"}

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/125.0 Safari/537.36")
pytrends = TrendReq(hl='el-GR', tz=120, timeout=(10, 30),
                    requests_args={'headers': {'User-Agent': UA}})
geo = 'GR'

ANCHORS = ["Nike", "Adidas"]
OTHERS = [
    "New Balance", "ASICS", "Puma", "Under Armour", "Reebok", "Saucony",
    "Jordan", "UGG", "Vans", "Hoka", "Birkenstock", "Salomon",
    "Converse", "Dr. Martens", "On", "Crocs", "Skechers",
]
QUERY_MAP = {"On": "On Cloud"}         # ασαφείς όροι -> query ή topic "/m/..."
def q(name): return QUERY_MAP.get(name, name)
OTHERS = [b for b in dict.fromkeys(OTHERS) if b not in ANCHORS]

MAX_RETRIES = 6
SLEEP_BETWEEN = (15, 25)

def fetch(display_names, timeframe):
    queries = [q(n) for n in display_names]
    back = {q(n): n for n in display_names}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            pytrends.build_payload(queries, timeframe=timeframe, geo=geo)
            df = pytrends.interest_over_time()
            if df.empty:
                raise RuntimeError("empty payload")
            return df.drop(columns=['isPartial'], errors='ignore').rename(columns=back)
        except Exception as e:
            is_rate = "429" in str(e) or "many" in str(e).lower() or "rate" in str(e).lower()
            base = 60 if is_rate else 15
            wait = min(300, base * (2 ** (attempt - 1))) + random.uniform(0, 10)
            print(f"    ⚠️ attempt {attempt}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES:
                print(f"    ⏳ cool-down {wait:.0f}s ..."); time.sleep(wait)
            else:
                raise

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def collect(label):
    timeframe = TIMEFRAMES[label]
    fname = f"Top_Brands_Direct_GR_{label}.csv"
    print(f"\n===== {label}  ({timeframe}) =====")

    # --- resume: φόρτωσε ό,τι έχει ήδη σωθεί ---
    if os.path.exists(fname):
        master = pd.read_csv(fname, index_col=0, parse_dates=True)
        print(f"  ↩️  resume — υπάρχουν ήδη: {list(master.columns)}")
    else:
        master = fetch(ANCHORS, timeframe)[ANCHORS].copy()   # reference scale
        master.index.name = "date"; master.to_csv(fname, encoding="utf-8-sig")
        time.sleep(random.uniform(*SLEEP_BETWEEN))

    ref_sum = master["Nike"].mean() + master["Adidas"].mean()   # η κλίμακα ζει στους anchors
    todo = [b for b in OTHERS if b not in master.columns]
    print(f"  Απομένουν: {todo if todo else 'κανένα'}")

    for grp in chunks(todo, 3):
        print(f"  Batch: {grp}")
        df = fetch(ANCHORS + grp, timeframe)
        nike_raw = df["Nike"].mean()
        for b in grp:
            if b in df.columns and df[b].mean() > nike_raw:
                print(f"    ⚠️ '{b}' > Nike στο batch — πιθανό squish. Βάλ' το στο QUERY_MAP.")
        factor = ref_sum / (nike_raw + df["Adidas"].mean())
        for b in grp:
            if b in df.columns:
                master[b] = (df[b] * factor).round(2).reindex(master.index)
        master.to_csv(fname, encoding="utf-8-sig")           # checkpoint μετά ΚΑΘΕ batch
        time.sleep(random.uniform(*SLEEP_BETWEEN))

    master = master[ANCHORS + [c for c in master.columns if c not in ANCHORS]]
    master.index.name = "date"; master.to_csv(fname, encoding="utf-8-sig")
    hi = [c for c in master.columns if c not in ANCHORS and master[c].mean() > master["Nike"].mean()]
    if hi: print(f"  ⚠️ Πάνω από Nike (έλεγξέ τα): {hi}")
    print(f"  💾 {fname} | {master.shape[1]} brands x {master.shape[0]} εβδομάδες")

    try:
        from google.colab import files
        files.download(fname)                                 # download ΑΜΕΣΩΣ ανά timeframe
    except Exception as e:
        print("  ℹ️ Κατέβασέ το από /content/ :", fname, "|", e)
    return fname

for lbl in RUN:
    collect(lbl)
print("\n🎉 Ολοκληρώθηκε:", RUN)

ModuleNotFoundError: No module named 'pytrends'

In [ ]:
import os, time, random
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta
from pytrends.request import TrendReq

# ============================================================
# ΤΕΛΙΚΟΣ ΚΩΔΙΚΑΣ — Brand collection (Google Trends, GR)  [RESUMABLE]
# Μέθοδος: STAR ANCHORING (Nike + Adidas anchors σε ΚΑΘΕ batch).
# Ανθεκτικό στα 429: checkpoint μετά από ΚΑΘΕ batch -> αν κολλήσει,
# απλά ΞΑΝΑΤΡΕΞΕ το cell και συνεχίζει από εκεί που έμεινε.
# ============================================================

# >>> Διάλεξε τι θα τρέξεις. Καλύτερα ΕΝΑ timeframe τη φορά. <<<
RUN = ["3y"]          # π.χ. ["1y"] ή ["3y"]
# 3y = ενιαίο παράθυρο 3 ετών (weekly, μία κλίμακα — ΟΧΙ ετήσια κομμάτια)
_end = date.today(); _start3 = _end - relativedelta(years=3)
TIMEFRAMES = {"1y": "today 12-m",
              "3y": f"{_start3} {_end}",     # custom range -> weekly, μία κλίμακα
              "5y": "today 5-y"}

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/125.0 Safari/537.36")
pytrends = TrendReq(hl='el-GR', tz=120, timeout=(10, 30),
                    requests_args={'headers': {'User-Agent': UA}})
geo = 'GR'

ANCHORS = ["Nike", "Adidas"]
OTHERS = [
    "New Balance", "ASICS", "Puma", "Under Armour", "Reebok", "Saucony",
    "Jordan", "UGG", "Vans", "Hoka", "Birkenstock", "Salomon",
    "Converse", "Dr. Martens", "On", "Crocs", "Skechers",
]
QUERY_MAP = {"On": "On Cloud"}         # ασαφείς όροι -> query ή topic "/m/..."
def q(name): return QUERY_MAP.get(name, name)
OTHERS = [b for b in dict.fromkeys(OTHERS) if b not in ANCHORS]

MAX_RETRIES = 6
SLEEP_BETWEEN = (15, 25)

def fetch(display_names, timeframe):
    queries = [q(n) for n in display_names]
    back = {q(n): n for n in display_names}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            pytrends.build_payload(queries, timeframe=timeframe, geo=geo)
            df = pytrends.interest_over_time()
            if df.empty:
                raise RuntimeError("empty payload")
            return df.drop(columns=['isPartial'], errors='ignore').rename(columns=back)
        except Exception as e:
            is_rate = "429" in str(e) or "many" in str(e).lower() or "rate" in str(e).lower()
            base = 60 if is_rate else 15
            wait = min(300, base * (2 ** (attempt - 1))) + random.uniform(0, 10)
            print(f"    ⚠️ attempt {attempt}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES:
                print(f"    ⏳ cool-down {wait:.0f}s ..."); time.sleep(wait)
            else:
                raise

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def collect(label):
    timeframe = TIMEFRAMES[label]
    fname = f"Top_Brands_Direct_GR_{label}.csv"
    print(f"\n===== {label}  ({timeframe}) =====")

    # --- resume: φόρτωσε ό,τι έχει ήδη σωθεί ---
    if os.path.exists(fname):
        master = pd.read_csv(fname, index_col=0, parse_dates=True)
        print(f"  ↩️  resume — υπάρχουν ήδη: {list(master.columns)}")
    else:
        master = fetch(ANCHORS, timeframe)[ANCHORS].copy()   # reference scale
        master.index.name = "date"; master.to_csv(fname, encoding="utf-8-sig")
        time.sleep(random.uniform(*SLEEP_BETWEEN))

    ref_sum = master["Nike"].mean() + master["Adidas"].mean()   # η κλίμακα ζει στους anchors
    todo = [b for b in OTHERS if b not in master.columns]
    print(f"  Απομένουν: {todo if todo else 'κανένα'}")

    for grp in chunks(todo, 3):
        print(f"  Batch: {grp}")
        df = fetch(ANCHORS + grp, timeframe)
        nike_raw = df["Nike"].mean()
        for b in grp:
            if b in df.columns and df[b].mean() > nike_raw:
                print(f"    ⚠️ '{b}' > Nike στο batch — πιθανό squish. Βάλ' το στο QUERY_MAP.")
        factor = ref_sum / (nike_raw + df["Adidas"].mean())
        for b in grp:
            if b in df.columns:
                master[b] = (df[b] * factor).round(2).reindex(master.index)
        master.to_csv(fname, encoding="utf-8-sig")           # checkpoint μετά ΚΑΘΕ batch
        time.sleep(random.uniform(*SLEEP_BETWEEN))

    master = master[ANCHORS + [c for c in master.columns if c not in ANCHORS]]
    master.index.name = "date"; master.to_csv(fname, encoding="utf-8-sig")
    hi = [c for c in master.columns if c not in ANCHORS and master[c].mean() > master["Nike"].mean()]
    if hi: print(f"  ⚠️ Πάνω από Nike (έλεγξέ τα): {hi}")
    print(f"  💾 {fname} | {master.shape[1]} brands x {master.shape[0]} εβδομάδες")

    try:
        from google.colab import files
        files.download(fname)                                 # download ΑΜΕΣΩΣ ανά timeframe
    except Exception as e:
        print("  ℹ️ Κατέβασέ το από /content/ :", fname, "|", e)
    return fname

for lbl in RUN:
    collect(lbl)
print("\n🎉 Ολοκληρώθηκε:", RUN)


===== 3y  (2023-07-02 2026-07-02) =====
    ⚠️ attempt 1/6: The request failed: Google returned a response with code 429
    ⏳ cool-down 65s ...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
grp = ["Converse", "Dr. Martens", "On"]
ref = fetch(ANCHORS, "today 12-m"); ref_sum = ref["Nike"].mean() + ref["Adidas"].mean()
df = fetch(ANCHORS + grp, "today 12-m")
factor = ref_sum / (df["Nike"].mean() + df["Adidas"].mean())
m = pd.read_csv("Top_Brands_Direct_GR_1y.csv", index_col=0).drop(columns=["ON"], errors="ignore")
for b in grp: m[b] = (df[b] * factor).round(2).values
m.to_csv("Top_Brands_Direct_GR_1y.csv", encoding="utf-8-sig")
from google.colab import files; files.download("Top_Brands_Direct_GR_1y.csv")
print("On mean:", round(m["On"].mean(), 1))

    ⚠️ attempt 1/6: The request failed: Google returned a response with code 429
    ⏳ cool-down 68s ...


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

On mean: 1.4
